# **Sydney T1 Duty-Free Shop — Synthetic Dataset**

> #### ⚠️ **Disclaimer**  
> This project is based on personal retail work experience and uses a **fully synthetic dataset built around a hypothetical scenario**.  All data is artificially generated — no real customer, transaction, or operational data has been used or reproduced.  
> *"This dataset was designed from first-hand retail experience to reflect real-world airport duty-free dynamics as accurately as possible."*

A portfolio dataset simulating the sales operations of a **duty-free gift shop at Sydney International Airport Terminal 1 (T1)**.  
- Designed to practise SQL analytics, data modelling, and Tableau visualisation — covering realistic retail patterns such as flight-linked transactions, nationality-based purchasing behaviour, and multi-type promotions.

| Item | Detail |
|---|---|
| **Setting** | Sydney Int'l Airport (T1) Duty-Free Gift Shop — Retail Transaction Data |
| **Period** | 1 January 2024 – 31 December 2024 (52 weeks) |
| **Tables** | 5 |
| **Total records** | 20,219 Transactions |
| **Output** | 5 CSV files + 1 SQLite database |
| **Reproducibility** | `random.seed(42)` fixed |

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta
import os
import sqlite3

In [2]:
fake = Faker()

random.seed(42)
np.random.seed(42)

In [3]:
OUTPUT_DIR = "duty_free_data"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    
    print(f"✨ '{OUTPUT_DIR}' created.")

os.makedirs(OUTPUT_DIR, exist_ok=True)

✨ 'duty_free_data' created.


## 1. COMMON CONSTANTS


- Nationality weights based on Sydney Airport Q4 2024 official traffic report

- Rank (traffic): AU(1) > CN(2) > NZ(3) > US(4) > GB(5) > KR(6) > IN(7) > JP(8) > PH(9) > CA(10)

- AU weight (12%) is lower than raw traffic share (22%) but accounts for:
    
    ① Naturalised AU citizens travelling back to their hometown (CN/KR/etc origin)

    — these travellers have similar purchase behaviour to their origin nationality

    ② Other AU travellers have low duty-free conversion (buy locally)

- Note: NZ weight (8%) kept lower than raw traffic share:

    ① NZ biosecurity law restricts honey & some food imports from AU (MPI regulations)

    ② Many NZ-AU travellers are residents who can buy the same products locally

In [4]:
NATIONALITIES      = ["AU", "CN", "NZ", "US", "GB", "KR", "IN", "JP", "PH", "CA", "SG", "TH"]
NATIONALITY_WEIGHT = [0.12, 0.20, 0.08, 0.10, 0.09, 0.09, 0.08, 0.07, 0.05, 0.04, 0.04, 0.04]

AGE_GROUPS = ["Under 20", "20s", "30s", "40s", "50s", "60+"]

CATEGORIES = [
    "Cosmetics", "Liquor", "Jewellery", "Souvenir",
    "Confectionery", "Apparel", "Honey", "Indigenous", "Tea"
]

PAYMENT_METHODS = [
    "Credit/Debit Card",  # physical card tap / swipe
    "Cash",               # notes and coins
    "AliPay",             # CN digital wallet (QR scan)
    "WeChat Pay",         # CN digital wallet (QR scan)
    "Digital Wallet",     # Apple Pay / Google Pay (NFC)
]

## 2. FLIGHT DEFINITIONS


Each entry : (flight_no, destination, airline, dep_hour, dep_min, weekdays)

- Flight numbers are fixed in real-world format (CX101, KE601 ...)
- Operating weekdays fixed per route  (0=Mon ~ 6=Sun)
- Northeast Asia  : morning to afternoon departures
- Southeast Asia  : morning departures
- Long-haul (Dubai/Doha/LA) : night departures

In [5]:
FLIGHT_DEF = [
    # ── Northeast Asia ──────────────────────────────────────
    ("CX101", "Hong Kong",    "Direct", "Cathay Pacific",        10, 30, [0,1,2,3,4,5,6]),  # daily
    ("CX103", "Hong Kong",    "Direct", "Cathay Pacific",        15, 45, [0,2,4,6]),         # Mon/Wed/Fri/Sun
    ("MU502", "Shanghai",     "Direct", "China Eastern",          9, 15, [0,1,3,4,6]),       # 5x/week
    ("CA781", "Beijing",      "Direct", "Air China",             10, 0,  [1,3,5]),            # 3x/week
    ("KE601", "Seoul",        "Direct", "Korean Air",            11, 30, [0,1,2,3,4,5,6]),  # daily
    ("OZ602", "Seoul",        "Direct", "Asiana Airlines",        9, 45, [0,2,4,5]),         # 4x/week
    ("JL771", "Tokyo",        "Direct", "Japan Airlines",        10, 0,  [0,1,2,3,4,5,6]),  # daily
    ("NH880", "Tokyo",        "Direct", "ANA",                   11, 15, [1,3,5,6]),         # 4x/week

    # ── Southeast Asia ──────────────────────────────────────
    ("SQ211", "Singapore",    "Direct", "Singapore Airlines",     8, 0,  [0,1,2,3,4,5,6]),  # daily
    ("SQ213", "Singapore",    "Direct", "Singapore Airlines",    13, 30, [0,2,3,5,6]),       # 5x/week
    ("TR13",  "Singapore",    "Direct", "Scoot",                  7, 45, [1,3,5]),            # 3x/week
    ("PR211", "Manila",       "Direct", "Philippine Airlines",    9, 30, [0,2,4,5,6]),       # 5x/week
    ("5J32",  "Manila",       "Direct", "Cebu Pacific",           8, 0,  [1,4,6]),            # 3x/week
    ("MH122", "Kuala Lumpur", "Direct", "Malaysia Airlines",      8, 45, [0,1,3,4,6]),       # 5x/week
    ("TG476", "Bangkok",      "Direct", "Thai Airways",           9, 0,  [0,2,3,5,6]),       # 5x/week
    ("GA715", "Bali",         "Direct", "Garuda Indonesia",       7, 30, [0,1,2,3,4,5,6]),  # daily

    # ── Long-haul (night departures) ────────────────────────
    ("QF1",   "Singapore",    "Direct", "Qantas",                16, 30, [0,1,2,3,4,5,6]),  # daily SYD→SIN (connects SIN→LHR as QF2)
    ("EK413", "Dubai",        "Direct", "Emirates",              21, 10, [0,1,2,3,4,5,6]),  # daily SYD→DXB
    ("QR007", "Doha",         "Direct", "Qatar Airways",         20, 45, [0,2,4,6]),          # 4x/week SYD→DOH
    ("QF7",   "Los Angeles",  "Direct", "Qantas",                11, 55, [0,1,2,3,4,5,6]),  # daily
    ("UA839", "Los Angeles",  "Direct", "United Airlines",       21, 0,  [1,3,5,6]),         # 4x/week
]

## 3. ITEM_MAP / SKU VARIANT DEFINITIONS / ITEM_PROMO / COST RATIOS


### 3.1 ITEMS

In [6]:
ITEM_MAP = {
    # ── Cosmetics ─────────────────────────────────────────
    # Item names used for PREF_MAP lookups; variants defined in COSMETICS_SKU_VARIANTS
    "Cosmetics": [
        "Lanolin Cream",
        "Sheep Placenta Cream",
        "Emu Oil Cream",
        "Paw Paw Ointment",
        "Wild Fern Skincare",
    ],
    # ── Liquor ────────────────────────────────────────────
    # Item names used for PREF_MAP lookups; variants defined in LIQUOR_SKU_VARIANTS
    "Liquor": [
        "Penfolds Wine",
        "Johnnie Walker Whisky",
        "Hennessy Cognac",
        "Absolut Vodka",
        "Tanqueray Gin",
    ],
    # ── Jewellery ─────────────────────────────────────────
    # AU/NZ specialty jewellery — opal, paua shell & jade
    "Jewellery": [
        "Opal Jewellery",        # AU opal — wide price range by grade
        "Paua Shell Jewellery",  # NZ paua shell — iridescent blue/green
        "Jade Jewellery",        # Australian/International/New Zealand jade 
    ],
    # ── Souvenir ──────────────────────────────────────────
    # Sydney / Australia themed gift items
    "Souvenir": [
        "SYD Keychain",
        "SYD Mug",
        "AUS Magnet",
        "Kangaroo Plush",
        "Koala Plush",
        "AUS Tea Towel",
    ],
    # ── Confectionery ─────────────────────────────────────
    # Branded chocolate + AU specialty snacks
    "Confectionery": [
        "Cadbury Chocolate",  # global chocolate (AU-produced)
        "Tim Tam Biscuit",    # AU iconic biscuit
        "Patons Chocolate",   # AU chocolate
        "Lindt Chocolate",    # global premium
        "Macadamia Nuts",     # AU specialty nut
        "Kangaroo Jerky",     # AU airport snack
    ],
    # ── Apparel ───────────────────────────────────────────
    # AU themed clothing & specialty items
    "Apparel": [
        "AUS Souvenir T-Shirt",
        "AUS Souvenir Hoodie",
        "Ugg Boots",
        "Ugg Slipper",
        "AUS Souvenir Cap",
    ],
    # ── Honey ─────────────────────────────────────────────
    # Brand names only — grade & price assigned per SKU in HONEY_SKU_VARIANTS
    # Each brand generates multiple SKUs (one per grade)
    "Honey": [
        "Comvita",       # 4 grades: UMF 5+ / 10+ / 15+ / 20+
        "Manuka Health", # 2 grades: MGO 263+ / 400+
        "Beepower",      # 2 grades: MGO 115+ / 263+
    ],
    # ── Indigenous ────────────────────────────────────────
    # Aboriginal art & AU native beauty products
    "Indigenous": [
        "Aboriginal Art Print",   # aboriginal art print
        "Boomerang",              # traditional boomerang
        "Indigenous Hand Cream",  # native hand cream
        "Indigenous Lip Balm",    # native lip balm
        "Indigenous Kitchenware", # aboriginal pattern kitchenware
    ],
    # ── Tea ───────────────────────────────────────────────
    # AU tea brands
    "Tea": [
        "T2 Tea"
    ]
}

### 3.2 SKU VARIANT DEFINITIONS


- All categories use variant-based SKU generation.
- Format: (Item, Variant, Price AUD)


In [7]:
# ── Cosmetics ─────────────────────────────────────────────────
COSMETICS_SKU_VARIANTS = [
    # Lanolin Cream
    ("Lanolin Cream", "A Brand 100ml",           8),
    ("Lanolin Cream", "B Brand 100ml",          12),
    ("Lanolin Cream", "A Brand 6-pack 100ml",   36),
    ("Lanolin Cream", "B Brand 6-pack 100ml",   60),
    # Sheep Placenta Cream
    ("Sheep Placenta Cream", "A Brand 100ml",         9),
    ("Sheep Placenta Cream", "B Brand 100ml Premium",22),
    ("Sheep Placenta Cream", "A Brand 6-pack 100ml", 45),
    # Emu Oil Cream
    ("Emu Oil Cream", "A Brand 100ml",           9),
    ("Emu Oil Cream", "A Brand 6-pack 100ml",   45),
    # Paw Paw Ointment
    ("Paw Paw Ointment", "L Paw Paw 75g",     10),
    ("Paw Paw Ointment", "G Paw Paw 75g",      9),
    # Wild Fern Skincare
    ("Wild Fern Skincare", "Lip Balm",          12),
    ("Wild Fern Skincare", "Hand Cream",        18),
    ("Wild Fern Skincare", "Face Cream",        27),
    ("Wild Fern Skincare", "Face Mask",         11),
    ("Wild Fern Skincare", "Gift Set",          42),
]

# ── Liquor ────────────────────────────────────────────────────
LIQUOR_SKU_VARIANTS = [
    # Penfolds Wine
    ("Penfolds Wine", "Max's",     30),
    ("Penfolds Wine", "Bin 407",  115),
    ("Penfolds Wine", "Bin 28",    35),
    ("Penfolds Wine", "Bin 389",  120),
    ("Penfolds Wine", "Grange",   999),
    # Johnnie Walker Whisky
    ("Johnnie Walker Whisky", "Red Label 700ml",   40),
    ("Johnnie Walker Whisky", "Black Label 700ml",  55),
    ("Johnnie Walker Whisky", "Gold Label 700ml",  100),
    # Hennessy Cognac
    ("Hennessy Cognac", "VS 700ml",    62),
    ("Hennessy Cognac", "VSOP 700ml",  88),
    ("Hennessy Cognac", "XO 700ml",   210),
    # Absolut Vodka
    ("Absolut Vodka", "Original 700ml",    42),
    ("Absolut Vodka", "Flavoured 700ml",   45),
    ("Absolut Vodka", "Elyx 700ml",        72),
    # Tanqueray Gin
    ("Tanqueray Gin", "London Dry 700ml",  48),
    ("Tanqueray Gin", "No. Ten 700ml",     65),
    ("Tanqueray Gin", "Rangpur 700ml",     52),
]

# ── Jewellery ─────────────────────────────────────────────────
JEWELLERY_SKU_VARIANTS = [
    # Opal Jewellery
    ("Opal Jewellery",       "Earrings",         85),
    ("Opal Jewellery",       "Necklace",        120),
    ("Opal Jewellery",       "Pendant",         220),
    ("Opal Jewellery",       "Premium Opal",    520),
    # Paua Shell Jewellery
    ("Paua Shell Jewellery", "Earrings",         38),
    ("Paua Shell Jewellery", "Pendant",          55),
    ("Paua Shell Jewellery", "Bracelet",         75),
    ("Paua Shell Jewellery", "Ring",             90),
    # Jade Jewellery
    ("Jade Jewellery",       "Pendant Small",    65),
    ("Jade Jewellery",       "Pendant Large",   110),
    ("Jade Jewellery",       "Bracelet",        150),
    ("Jade Jewellery",       "Ring",            200),
]

# ── Souvenir ──────────────────────────────────────────────────
SOUVENIR_SKU_VARIANTS = [
    # SYD Keychain
    ("SYD Keychain",   "Metal",          12),
    ("SYD Keychain",   "Acrylic",         9),
    # SYD Mug
    ("SYD Mug",        "Standard 350ml", 20),
    ("SYD Mug",        "Large 500ml",    25),
    # AUS Magnet
    ("AUS Magnet",     "Standard",       10),
    ("AUS Magnet",     "3-Pack",         18),
    # Kangaroo Plush
    ("Kangaroo Plush", "Small 20cm",     18),
    ("Kangaroo Plush", "Large 40cm",     45),
    # Koala Plush
    ("Koala Plush",    "Small 20cm",     18),
    ("Koala Plush",    "Large 40cm",     45),
    # AUS Tea Towel
    ("AUS Tea Towel",  "Single",         18),
]

# ── Confectionery ─────────────────────────────────────────────
CONFECTIONERY_SKU_VARIANTS = [
    # Cadbury Chocolate (3 SKUs)
    ("Cadbury Chocolate", "Milk Chocolate 200g",   9),
    ("Cadbury Chocolate", "Gift Box Assorted",     16),
    ("Cadbury Chocolate", "Favourites Gift Box",   22),
    # Tim Tam — flavour variants
    ("Tim Tam Biscuit", "Original",                7),
    ("Tim Tam Biscuit", "Dark Chocolate",          7),
    ("Tim Tam Biscuit", "White",                   7),
    ("Tim Tam Biscuit", "Double Coat",             7),
    ("Tim Tam Biscuit", "Gluten Free",             7),
    ("Tim Tam Biscuit", "Chewy Caramel",           7),
    ("Tim Tam Biscuit", "Jatz",                    7),
    # Patons Chocolate (3 SKUs)
    ("Patons Chocolate", "Milk Chocolate 200g",   22),
    ("Patons Chocolate", "Dark Chocolate 200g",   22),
    ("Patons Chocolate", "Gift Box",              27),
    # Lindt Chocolate (3 SKUs)
    ("Lindt Chocolate", "Excellence 70% 100g",    12),
    ("Lindt Chocolate", "Lindor Box 200g",        22),
    ("Lindt Chocolate", "Gift Box Assorted",      35),
    # Macadamia Nuts (3 SKUs)
    ("Macadamia Nuts", "Honey Roasted 100g",      13),
    ("Macadamia Nuts", "Mixed Flavour 200g",      27),
    ("Macadamia Nuts", "Gift Tin 300g",           36),
    # Kangaroo Jerky — weight variants
    ("Kangaroo Jerky", "Original 50g",            11),
    ("Kangaroo Jerky", "Original 100g",           17),
]

# ── Apparel ───────────────────────────────────────────────────
APPAREL_SKU_VARIANTS = [
    # AUS Souvenir T-Shirt
    ("AUS Souvenir T-Shirt", "Kids",      28),
    ("AUS Souvenir T-Shirt", "Adult",     40),
    # AUS Souvenir Hoodie
    ("AUS Souvenir Hoodie",  "Standard",  55),
    ("AUS Souvenir Hoodie",  "Premium",   80),
    # Ugg Boots
    ("Ugg Boots",            "Classic Short", 110),
    ("Ugg Boots",            "Classic Tall",  180),
    # Ugg Slipper
    ("Ugg Slipper",          "Slipper",       100),
    # AUS Souvenir Cap
    ("AUS Souvenir Cap",     "Standard",   18),
    ("AUS Souvenir Cap",     "Premium",    27),
]

# ── Honey ─────────────────────────────────────────────────────
HONEY_SKU_VARIANTS = [
    ("Comvita",       "UMF 5+",    45),
    ("Comvita",       "UMF 10+",   60),
    ("Comvita",       "UMF 15+",   90),
    ("Comvita",       "UMF 20+",  170),
    ("Manuka Health", "MGO 263+",  55),
    ("Manuka Health", "MGO 400+",  75),
    ("Beepower",      "MGO 115+",  35),
    ("Beepower",      "MGO 263+",  45),
]

# ── Indigenous ────────────────────────────────────────────────
INDIGENOUS_SKU_VARIANTS = [
    # Aboriginal Art Print
    ("Aboriginal Art Print",   "A4 Print",       28),
    ("Aboriginal Art Print",   "A3 Print",       55),
    ("Aboriginal Art Print",   "Framed A3",     110),
    # Boomerang
    ("Boomerang",              "Painted Small",  25),
    ("Boomerang",              "Painted Large",  55),
    # Indigenous Hand Cream
    ("Indigenous Hand Cream",  "50ml",           14),
    ("Indigenous Hand Cream",  "100ml",          24),
    # Indigenous Lip Balm
    ("Indigenous Lip Balm",    "Single",          9),
    ("Indigenous Lip Balm",    "3-Pack",         18),
    # Indigenous Kitchenware
    ("Indigenous Kitchenware", "Coaster Set",    22),
    ("Indigenous Kitchenware", "Cutting Board",  48),
]

# ── Tea ───────────────────────────────────────────────────────
# T2 only — Dilmah / Twinings / Madura removed per design decision
TEA_SKU_VARIANTS = [
    ("T2 Tea", "Small Tin 50g",  22),
    ("T2 Tea", "Large Tin 100g", 42),
    ("T2 Tea", "Gift Set",       62),
]

CN customers show strong preference for high-value items:

① Honey : UMF 10+ / MGO 400+ and above (gift-quality grades)

② Liquor: $85+ bottles (Hennessy VSOP/XO, JW Gold, Penfolds Bin 407/389/Grange)

In [8]:
# ── CN nationality-based SKU preference ──────────────────────
CN_PREMIUM_HONEY_VARIANTS   = ["UMF 10+", "UMF 15+", "UMF 20+", "MGO 400+"]
CN_PREMIUM_LIQUOR_MIN_PRICE = 85.0
CN_PREMIUM_TRIGGER_RATE     = 0.70

In [9]:
# PRICE_RANGE — fallback only (not reached in normal execution)
# Reflects actual min/max across all variants in each category's SKU_VARIANTS list
PRICE_RANGE = {
    "Cosmetics"    : (8,    42),  # A Brand 100ml Lanolin Cream $8 → Wild Fern Gift Set $42
    "Liquor"       : (30,  999),  # Penfolds Max's $30 → Grange $999
    "Jewellery"    : (38,  520),  # Paua Earrings $38 → Opal Premium $520
    "Souvenir"     : (9,    45),  # AUS Magnet Acrylic $9 → Kangaroo Plush Large $45
    "Confectionery": (7,    36),  # Tim Tam $7 → Macadamia Gift Tin $36
    "Apparel"      : (22,  180),  # AUS Cap Standard $22 → Ugg Classic Tall $180
    "Honey"        : (35,  170),  # Beepower MGO 115+ $35 → Comvita UMF 20+ $170
    "Indigenous"   : (9,   110),  # Indigenous Lip Balm Single $9 → Aboriginal Art Framed A3 $110
    "Tea"          : (22,   62),  # T2 Small Tin $22 → T2 Gift Set $62
}

### 3.3 Item-level promotions

- These run year-round (not tied to holidays)
- Format: (promo_id, description, category, item_filter, promo_type, params)

- Promo types used:
    - BUY_X_GET_Y_ITEM  : buy X of matching item, get 1 free (same item)
    - BUNDLE_FIXED_ITEM : N of matching item for fixed price
    - MIX_BUY_X_GET_Y   : buy X from category pool, get 1 free (random SKU, paid qty only)

In [10]:
ITEM_PROMOS = [
    # Cosmetics — Paw Paw Ointment: Buy 5 Get 6th Free
    # G Paw Paw ($9) and L Paw Paw ($10) both eligible — mix allowed
    # The 6th (free) item is always G Paw Paw ($9) — cheapest of the two
    # free_item_price : retail price of the free item (excluded from Total_Amount)
    # free_item_cost  : cost of the free item (included in Total_Cost for margin accuracy)
    {
        "promo_id"        : "IP-01",
        "description"     : "Paw Paw Ointment Buy 5 Get 6th Free (G Paw Paw free)",
        "category"        : "Cosmetics",
        "item_filter"     : "Paw Paw Ointment",
        "variant_filter"  : None,               # both G and L eligible to buy
        "promo_type"      : "MIX_BUY_X_GET_Y",
        "buy_qty"         : 5,
        "free_qty"        : 1,
        "bundle_price"    : None,
        "free_item_price" : 9.0,                # G Paw Paw retail price
        "free_item_cost"  : round(9.0 * 0.40, 2),  # G Paw Paw cost ($3.60)
    },
    # Honey — Comvita UMF 5+ only: Buy 3 Get 4th Free
    {
        "promo_id"   : "IP-02",
        "description": "Comvita UMF 5+ Buy 3 Get 4th Free",
        "category"   : "Honey",
        "item_filter": "Comvita",             # Item == "Comvita", Variant == "UMF 5+"
        "variant_filter": "UMF 5+",
        "promo_type" : "BUY_X_GET_Y_ITEM",
        "buy_qty"    : 3,
        "free_qty"   : 1,
        "bundle_price": None,
    },
    # Confectionery — Tim Tam any flavour: 4 for $24
    # All Tim Tam SKUs are $7 → 4×$7=$28 → bundle $24 saves $4
    {
        "promo_id"   : "IP-03",
        "description": "Tim Tam Any Flavour 4 for $24",
        "category"   : "Confectionery",
        "item_filter": "Tim Tam Biscuit",
        "variant_filter": None,               # any Tim Tam flavour
        "promo_type" : "BUNDLE_FIXED_ITEM",
        "buy_qty"    : 4,
        "free_qty"   : 0,
        "bundle_price": 24.0,
    },
]

In [11]:
# Probability that an eligible transaction triggers a promo
# (not every customer buying that item will take the promo deal)
PROMO_TRIGGER_RATE = {
    "IP-01": 0.20,   # 20% of Paw Paw buyers take the 6-pack deal
    "IP-02": 0.25,   # 25% of Comvita UMF 5+ buyers buy 4
    "IP-03": 0.40,   # 30% of Tim Tam buyers take the 4-pack deal
}

### 3.4 Cost Ratio

Cost_Price = Selling_Price × COST_RATIO

In [12]:
# ── Cost ratios ───────────────────────────────────────────────
COST_RATIO = {
    "Cosmetics"    : 0.45,
    "Liquor"       : 0.65,
    "Jewellery"    : 0.60,
    "Souvenir"     : 0.45,
    "Confectionery": 0.55,
    "Apparel"      : 0.50,
    "Honey"        : 0.60,
    "Indigenous"   : 0.50,
    "Tea"          : 0.50,
}

## 4. NATIONALITY × PREFERRED CATEGORY  (weighted tuples)


- Weights do NOT need to sum to 100.
- pick_category() distributes any remainder evenly across
- unlisted categories, so every category has some purchase probability.
- Listed categories = strong preferences; remainder = background noise.

In [13]:
PREF_MAP = {
    # AU: lower duty-free conversion than raw traffic share
    # duty-free price benefit (main driver)
    # snacks/gifts for family & friends
    # Paw Paw, Lanolin — pharmacy substitute at airport
    "AU": [("Liquor",15),("Confectionery",25),("Cosmetics",10),("Souvenir",10)],

    # CN: manuka honey + cosmetics + cognac + souvenirs (gift-buying culture)
    "CN": [("Honey",30),("Cosmetics",26),("Liquor",16),("Confectionery",12),("Souvenir",16)],

    # NZ: souvenirs + confectionery + tea
    # Honey very low — NZ MPI biosecurity restricts AU honey imports
    "NZ": [("Souvenir",30),("Confectionery",25),("Tea",20),("Cosmetics",10),("Honey",5)],

    # US: apparel + indigenous + souvenir + jewellery
    # Liquor reduced — not a strong duty-free driver for US travellers at AU airports
    "US": [("Apparel",28),("Indigenous",24),("Souvenir",20),("Jewellery",12)],

    # GB: tea + apparel + indigenous
    # Liquor reduced — not a strong duty-free driver for GB travellers at AU airports
    "GB": [("Tea",30),("Apparel",24),("Indigenous",18),("Souvenir",12)],

    # KR: cosmetics + honey + wine (occasional) + confectionery
    # Honey and confectionery bought at similar rates
    "KR": [("Cosmetics",35),("Honey",22),("Confectionery",20),("Liquor",10),("Jewellery",8)],

    # IN: confectionery + souvenir + cosmetics
    "IN": [("Confectionery",32),("Souvenir",25),("Tea",18),("Cosmetics",10)],

    # JP: confectionery + souvenir + tea (omiyage gift culture)
    "JP": [("Confectionery",38),("Souvenir",28),("Tea",18),("Cosmetics",8)],

    # PH: confectionery + souvenir + cosmetics
    "PH": [("Confectionery",35),("Souvenir",30),("Cosmetics",18),("Tea",8)],

    # CA: apparel + indigenous + souvenir
    # Liquor reduced — similar to US/GB pattern
    "CA": [("Apparel",28),("Indigenous",24),("Souvenir",20),("Tea",10)],

    # SG: cosmetics + confectionery + honey
    "SG": [("Cosmetics",30),("Confectionery",24),("Honey",18),("Tea",12),("Souvenir",8)],

    # TH: cosmetics + souvenir + confectionery
    "TH": [("Cosmetics",32),("Souvenir",26),("Confectionery",20),("Tea",8)],
}

In [14]:
DEST_CAT_BIAS = {
    # Northeast Asia routes: Honey & Cosmetics are strong sellers (CN/KR/JP travellers)
    "asia_east": ["Cosmetics","Honey","Confectionery","Tea"],
    # Southeast Asia routes: Souvenir & Confectionery dominant
    "asia_se"  : ["Souvenir","Confectionery","Cosmetics","Tea"],
    # Long-haul routes: Liquor, Apparel, Jewellery, Indigenous art
    "long_haul": ["Liquor","Apparel","Jewellery","Indigenous"],
}

In [15]:
DEST_GROUP = {
    # Northest Asia
    "Hong Kong":"asia_east",
    "Shanghai":"asia_east",
    "Beijing":"asia_east",
    "Seoul":"asia_east",
    "Tokyo":"asia_east",
    # Southeast Asia
    "Singapore":"asia_se",
    "Manila":"asia_se",
    "Kuala Lumpur":"asia_se",
    "Bangkok":"asia_se",
    "Bali":"asia_se",
    # Long-haul
    "Dubai":"long_haul",
    "Doha":"long_haul",
    "Los Angeles":"long_haul",
}

In [16]:
def pick_category(nat: str, dest: str, active_holiday: dict = None) -> str:
    """
    Resolve purchase category using two-stage weighted selection.

    Stage 1 (70% probability) — nationality preference:
      Uses PREF_MAP weights. If total < 100, the remainder is
      distributed evenly across unlisted categories so every
      category has some non-zero purchase probability.
      If a holiday is active, the target category receives an
      additional Boost_Weight on top of its normal weight.

    Stage 2 (30% probability) — destination group bias:
      Uses DEST_CAT_BIAS for the flight's destination group.
    """
    if random.random() < 0.70:
        pref_opts = PREF_MAP.get(nat, [])
        listed_cats   = {c for c, _ in pref_opts}
        listed_weight = sum(w for _, w in pref_opts)
        remainder     = max(0, 100 - listed_weight)

        # Distribute remainder evenly across unlisted categories
        unlisted_cats = [c for c in CATEGORIES if c not in listed_cats]
        if unlisted_cats:
            base_weight = remainder / len(unlisted_cats)
        else:
            base_weight = 0

        all_cats    = [c for c, _ in pref_opts] + unlisted_cats
        all_weights = [w for _, w in pref_opts] + [base_weight] * len(unlisted_cats)

        # Apply holiday boost — add extra weight to target category
        if active_holiday and active_holiday.get("cat"):
            boost_cat = active_holiday["cat"]
            boost_val = active_holiday.get("boost", 1.2)
            
            if boost_cat in all_cats:
                idx = all_cats.index(boost_cat)
                all_weights[idx] += (all_weights[idx] * (boost_val - 1))

        if sum(all_weights) <= 0:
            return random.choice(CATEGORIES)

        return random.choices(all_cats, weights=all_weights, k=1)[0]

    else:
        group = DEST_GROUP.get(dest)
        pool  = DEST_CAT_BIAS.get(group, CATEGORIES) if group else CATEGORIES
        return random.choice(pool)


## 5. Table 1 — customer_details (8,000 customers)

In this dataset, I have intentionally simulated **"Thin Profiles"** for approximately 13% of the customer base to reflect real-world data challenges in a high-traffic Duty-Free environment.

* **Definition:** Customer records that contain only a `customer_id`, while all other demographic fields (Nationality, Age Group, etc.) are set to `NULL`.
* **Business Context:** This reflects actual scenarios at **Sydney Airport Terminal 1**, where demographic data is captured via boarding pass scans. Incomplete profiles typically occur **possibly** due to:
    1. **Technical Scanning Failures:** Difficulty in reading mobile E-boarding passes due to screen glare or low brightness.
    2. **Transit Complexity:** Passengers in transit who may not yet have their connecting boarding pass or whose itinerary data is fragmented.
    3. **Operational Efficiency:** Manual entry overrides performed by staff during peak periods to maintain service speed.
* **Analytical Purpose:** To test the **robustness of SQL queries**. This forces the use of advanced handling for incomplete records (e.g., `COALESCE`, `IS NULL` filters) to ensure sales reports remain accurate and representative despite missing dimensions.

In [17]:
def generate_customers(n: int = 8000) -> pd.DataFrame:
    records = []
    for i in range(1, n + 1):
        # No Boarding Pass
        has_boarding_pass = random.random() < 0.13 

        if not has_boarding_pass:
            records.append({
                "customer_id"       : f"C-{i:05d}",
                "nationality"       : None,
                "age_group"         : None,
            })
        else:
            # Passengers
            records.append({
                "customer_id": f"C-{i:05d}",
                "nationality": random.choices(NATIONALITIES, weights=NATIONALITY_WEIGHT)[0],
                "age_group": random.choice(AGE_GROUPS)              
            })
    df = pd.DataFrame(records)
    
    df.columns = [c.lower() for c in df.columns]
    return df   


## 6. Table 2 — product_master  (10 SKUs per category, 108 total)


In [18]:
def generate_products() -> pd.DataFrame:
    """
    Build product_master — one row per SKU variant tuple.
    All categories use variant-based generation (no range-based random).

    Category      Variant list                  SKUs
    ──────────    ──────────────────────────    ────
    Cosmetics   → COSMETICS_SKU_VARIANTS         16
    Liquor      → LIQUOR_SKU_VARIANTS            17
    Jewellery   → JEWELLERY_SKU_VARIANTS         12
    Souvenir    → SOUVENIR_SKU_VARIANTS          11
    Confectionery→ CONFECTIONERY_SKU_VARIANTS    21
    Apparel     → APPAREL_SKU_VARIANTS            9
    Honey       → HONEY_SKU_VARIANTS              8
    Indigenous  → INDIGENOUS_SKU_VARIANTS        11
    Tea         → TEA_SKU_VARIANTS                3
    ──────────────────────────────────────────────
    Total                                       108
    """
    records, sku_n = [], 1

    VARIANT_MAP = {
        "Cosmetics"    : COSMETICS_SKU_VARIANTS,
        "Liquor"       : LIQUOR_SKU_VARIANTS,
        "Jewellery"    : JEWELLERY_SKU_VARIANTS,
        "Souvenir"     : SOUVENIR_SKU_VARIANTS,
        "Confectionery": CONFECTIONERY_SKU_VARIANTS,
        "Apparel"      : APPAREL_SKU_VARIANTS,
        "Honey"        : HONEY_SKU_VARIANTS,
        "Indigenous"   : INDIGENOUS_SKU_VARIANTS,
        "Tea"          : TEA_SKU_VARIANTS,
    }

    for cat in CATEGORIES:
        for (item, variant, price) in VARIANT_MAP[cat]:
            records.append({
                "product_sku"  : f"SKU-{sku_n:03d}",
                "category"     : cat,
                "item"         : item,
                "variant"      : variant,
                "selling_price": float(price),
                "cost_price"   : round(float(price) * COST_RATIO[cat], 2),
            })
            sku_n += 1

    df = pd.DataFrame(records)
    
    df.columns = [c.lower() for c in df.columns]
    return df

## 7. Table 3 — flight_schedules

- Key design : fixed flight numbers + fixed operating weekdays

    e.g. CX101 flies daily, MU502 flies Mon/Tue/Thu/Fri/Sun

- mirrors real-world airline schedule structure

- ±5 min random delay added to departure time for realism

> Note on Data Scope (via):
While many passengers from Sydney Airport have transit itineraries (e.g., via Singapore or Dubai), this dataset focuses on the immediate next destination as captured by the primary data source (Boarding Pass Scan). To ensure data integrity and avoid speculative bias, final destination (via) information is excluded from the analysis, as it is not recorded in the automated system logs and would otherwise require manual inquiry.

In [19]:
def generate_flights(weeks: int = 52) -> pd.DataFrame:
    records   = []
    base_date = datetime(2024, 1, 1)   # 2024-01-01 = Monday (full year: Jan–Dec 2024)

    for (fno, dest, _, airline, dep_h, dep_m, weekdays) in FLIGHT_DEF:
        for week in range(weeks):
            for wd in weekdays:
                # base_date (Monday) + week offset + weekday offset
                dep_time = (base_date
                            + timedelta(weeks=week, days=wd,
                                        hours=dep_h, minutes=dep_m)
                            + timedelta(minutes=random.randint(-5, 5)))  # ±5 min random delay
                # Flight_Status: 85% On Time, 13% Delayed, 2% Cancelled
                # Delayed flights add extra dwell time → potential sales boost
                status_roll = random.random()
                if status_roll < 0.85:
                    flight_status = "On Time"
                elif status_roll < 0.98:
                    flight_status = "Delayed"
                else:
                    flight_status = "Cancelled"

                records.append({
                    "flight_no"     : fno,
                    "airline"       : airline,
                    "destination"   : dest,
                    "departure_time": dep_time,
                    "flight_status" : flight_status,
                })

    df = pd.DataFrame(records)
    df = df.sort_values("departure_time").reset_index(drop=True)
    df.columns = [c.lower() for c in df.columns]
    
    return df

## 8. Table 4 — holiday_events  (Jan–Dec, 6 holidays)

- Purpose: define holiday periods and their target category.
- No price discounts — the shop does not run formal promotions.
- Instead, purchase patterns shift during holidays:
    travellers buy more of certain items around key events
    (e.g. more Honey around Lunar New Year, 
    more Confectionery around Easter).

Columns:
- event_id         : unique identifier
- event_name       : holiday or seasonal event name
- start_date       : start of holiday window
- end_date         : end of holiday window

In [20]:
def generate_holiday_events() -> pd.DataFrame:
    # (name, start, end, target_category, boost_weight, nat_boost, nat_weight)
    events = [
        # ── January ─────────────────────────────────────────────
        ("New Year Kickoff",      "2024-01-01", "2024-01-07", "Confectionery", 20,
         None, None),
        # ── February ────────────────────────────────────────────
        # Lunar New Year: CN/KR visitor surge
        ("Lunar New Year",        "2024-02-08", "2024-02-18", "Honey",         30,
         "CN,KR", "0.38,0.14"),
        # ── March / April ───────────────────────────────────────
        # Easter: AU/GB visitor surge
        ("Easter Long Weekend",   "2024-03-29", "2024-04-01", "Confectionery", 10,
         "AU,GB", "0.25,0.14"),
        # ── September ───────────────────────────────────────────
        # Korean Chuseok: KR visitor surge — major outbound travel
        # 2024 Chuseok: Sep 16-18 (Mon-Wed), long weekend Sep 14-18
        ("Chuseok",               "2024-09-13", "2024-09-19", "Souvenir", 20,
         "KR", "0.18"),
        # Chinese Mid-Autumn Festival: CN visitor surge
        # 2024 Mid-Autumn: Sep 15-17 (Sun-Tue)
        ("Mid-Autumn Festival",   "2024-09-14", "2024-09-18", "Honey",         20,
         "CN", "0.35"),
        # ── December ────────────────────────────────────────────
        # Christmas & New Year: AU/GB/US surge — holiday season travel peak
        ("Christmas & New Year",  "2024-12-22", "2024-12-31", "Souvenir", 20,
         None, None),
    ]

    records = []
    for i, (name, start, end, cat, boost, nat_boost, nat_weight) in enumerate(events, 1):
        records.append({
            "event_id": f"E-{i:02d}",
            "event_name": name,
            "start_date": start,
            "end_date": end,
            "target_category": cat,
            "boost_weight": boost,
            "nat_boost": nat_boost,
            "nat_weight": nat_weight
        })

    full_df = pd.DataFrame(records)
    
    public_cols = ["event_id", "event_name", "start_date", "end_date"]

    return full_df[public_cols] 

## 9. Table 5 — transactions  (10,000 records)


transaction time = 30 min–3 hrs before departure
1. Collect all flights departing on the chosen date
2. Pick one flight, then calculate purchase time within pre-departure window
3. Purchase time must fall within operating hours (06:00–23:00)

In [ ]:
def generate_transactions(
    customers_df: pd.DataFrame,
    products_df: pd.DataFrame,
    flights_df: pd.DataFrame,
    holiday_events_df: pd.DataFrame,
    n: int = 10000,
) -> pd.DataFrame:
    
    n = int(n)
    
    # ── 1. Lookup Preparation (Lowercasing & Indexing) ───────────────────
    cust_df = customers_df.copy()
    prod_df = products_df.copy()
    prod_df["cat_match"] = prod_df["category"].str.lower()

    # Pre-calculate sampling weights for price-based probability
    prod_df['sampling_weight'] = 1 / (prod_df['selling_price'] ** 0.8)

    cust_lookup = cust_df.set_index("customer_id")
    sku_df      = prod_df.set_index("product_sku") 
    
    cat_skus = {}
    cat_weights = {}

    for cat in CATEGORIES:
        mask = prod_df["cat_match"] == cat.lower()
        cat_skus[cat] = prod_df.loc[mask, "product_sku"].tolist()
        cat_weights[cat] = prod_df.loc[mask, "sampling_weight"].tolist()

    # Flight & Holiday Pre-processing (Ensure lowercase for flight columns)
    flights_df["_dep_dt"] = pd.to_datetime(flights_df["departure_time"])
    flights_df["_date"] = flights_df["_dep_dt"].dt.date
    
    date_to_flights = (
        flights_df.groupby("_date")
                  .apply(lambda g: g[["flight_no","destination","_dep_dt"]].to_dict("records"),
                         include_groups=False)
                  .to_dict()
    )
    all_dates = sorted(date_to_flights.keys())

    # Holiday setup (keeping your event logic)
    holiday_lookup = []
    holiday_lookup = []
    for _, r in holiday_events_df.iterrows():
        holiday_lookup.append({
            "event_id": r["event_id"], 
            "start": pd.to_datetime(r["start_date"]), 
            "end": pd.to_datetime(r["end_date"]),
        })

    # ── 2. Helper: pick_sku_and_amounts ──────────────────────────────────
    # Returns: (sku, unit_price, qty, original_price, final_price, line_cost, promo_id)
    def pick_sku_and_amounts(nat, category, sku_pool, current_weights):
        if not sku_pool:
            fallback_sku = random.choice(prod_df["product_sku"].tolist())
            p = sku_df.loc[fallback_sku]
            return fallback_sku, p["selling_price"], 1, p["selling_price"], p["selling_price"], p["cost_price"], None
        
        matched_promo = None
        for ip in [p for p in ITEM_PROMOS if p["category"] == category]:
            if random.random() < PROMO_TRIGGER_RATE.get(ip["promo_id"], 0.2):
                matched_promo = ip; break

        if matched_promo is None:
            # Price-weighted sampling instead of pure random.choice
            if nat == "CN" and category == "Honey" and random.random() < CN_PREMIUM_TRIGGER_RATE:
                pool = [s for s in sku_pool if sku_df.loc[s, "variant"] in CN_PREMIUM_HONEY_VARIANTS]
                sku = random.choice(pool) if pool else random.choices(sku_pool, weights=current_weights)[0]
            elif nat == "CN" and category == "Liquor" and random.random() < CN_PREMIUM_TRIGGER_RATE:
                pool = [s for s in sku_pool if sku_df.loc[s, "selling_price"] >= CN_PREMIUM_LIQUOR_MIN_PRICE]
                sku = random.choice(pool) if pool else random.choices(sku_pool, weights=current_weights)[0]
            else:
                sku = random.choices(sku_pool, weights=current_weights)[0]

            prod = sku_df.loc[sku]
            u_p = prod["selling_price"]
            qty_range = list(range(1, 21))

            if nat == "CN":
                cn_weights = [
                    25, 20, 15, 10, 8, 5,   # 1~6
                    3, 3, 2, 2,             # 7~10
                    1, 1, 1, 1, 1,          # 11~15
                    0.5, 0.5, 0.5, 0.5, 0.5 # 16~20
                ]
                qty = random.choices(qty_range, weights=cn_weights)[0]
            else:
                gen_weights = [
                    20, 20, 12, 16, 4,       # 1~5
                    2, 1.5, 1.2, 3, 0.5,     # 6~10
                    0.4, 0.3, 0.3, 0.2, 0.2, # 11~15
                    0.1, 0.1, 0.1, 0.1, 0.3  # 16~20
                ]
                qty = random.choices(qty_range, weights=gen_weights)[0]
            
            if u_p >= 500:  
                qty = random.choices([1, 2], weights=[90, 10])[0]

            return sku, u_p, qty, round(u_p*qty, 2), round(u_p*qty, 2), None

        else:
            ip = matched_promo
            eligible = [s for s in sku_pool if sku_df.loc[s, "item"] == ip["item_filter"]]
            if ip.get("variant_filter"):
                eligible = [s for s in eligible if sku_df.loc[s, "variant"] == ip["variant_filter"]]
            
            if not eligible: # Fallback
                sku = random.choices(sku_pool, weights=current_weights)[0]
                prod = sku_df.loc[sku]
                return sku, u_p, 1, u_p, u_p, None

            sku = random.choice(eligible)
            prod = sku_df.loc[sku]
            u_p = prod["selling_price"]

            if ip["promo_type"] == "BUY_X_GET_Y_ITEM":
                qty = ip["buy_qty"] + ip["free_qty"]
                return sku, u_p, qty, round(u_p*qty, 2), round(u_p*ip["buy_qty"], 2), ip["promo_id"]
            elif ip["promo_type"] == "BUNDLE_FIXED_ITEM":
                qty = ip["buy_qty"]
                return sku, u_p, qty, round(u_p*qty, 2), round(ip["bundle_price"], 2), ip["promo_id"]
            else:
                return sku, u_p, 1, u_p, u_p, ip["promo_id"]

    # ── 3. Main Generation Loop ──────────────────────────────────────────
    all_transactions = []
    tx_count, attempts = 0, 0
    cust_ids = cust_df["customer_id"].tolist()

    while tx_count < n and attempts < n * 15:
        attempts += 1
        date = random.choice(all_dates)
        day_flights = date_to_flights.get(date, [])
        if not day_flights: continue

        flight = random.choice(day_flights)
        if flight.get("flight_status") == "Cancelled": continue

        dep_dt, dest = flight["_dep_dt"], flight["destination"]
        trans_dt = dep_dt - timedelta(minutes=random.randint(30, 180))
        if not (6 <= trans_dt.hour < 23): continue

        active_holiday = next((h for h in holiday_lookup if h["start"].date() <= trans_dt.date() <= h["end"].date()), None)

        # Customer & Demographic logic
        cust_id = random.choice(cust_ids) 
        customer = cust_lookup.loc[cust_id]
        nat, age_grp = customer["nationality"], customer["age_group"]

        if nat == "AU" and random.random() < 0.8:
            continue

        # Basket Size
        if nat == "CN":
            n_items = random.choices([1, 2, 3, 4, 5], weights=[20, 15, 15, 35, 15])[0]
        elif nat in ["KR", "US", "GB"]:
            n_items = random.choices([1, 2, 3, 4], weights=[30, 45, 20, 5])[0]
        else:
            n_items = random.choices([1, 2, 3, 4], weights=[30, 45, 20, 5])[0]

        tx_id = f"T-{tx_count+1:06d}"

        pay_map = { 
            "AU": [50, 10,  0,  0, 40], "CN": [10, 18, 18, 18, 36],
            "NZ": [48, 10,  0,  0, 42], "US": [48, 15,  0,  0, 51],
            "GB": [55, 15,  0,  0, 30], "KR": [66, 18,  1,  0, 15],
            "IN": [20, 38,  0,  0, 42], "JP": [25, 30,  0,  0, 45],
            "PH": [20, 38,  0,  0, 42], "CA": [48,  8,  0,  0, 44],
            "SG": [45, 10,  2,  1, 42], "TH": [38, 32,  0,  0, 30],
        }
        weights = pay_map.get(nat, [40, 30, 0, 0, 30])
        
        payment = random.choices(PAYMENT_METHODS, weights=weights, k=1)[0]

        for line_no in range(1, n_items + 1):
            category = pick_category(nat, dest, active_holiday)
            sku_pool = cat_skus.get(category, prod_df["product_sku"].tolist())
            sku_w = cat_weights.get(category, [1] * len(sku_pool))

            current_weights = list(sku_w)

            for i, s in enumerate(sku_pool):
                is_premium = sku_df.loc[s, "selling_price"] > 100
                if nat == "KR" and category in ["Honey", "Liquor"] and is_premium:
                    current_weights[i] *= 2
                elif nat in ["US", "GB"] and category in ["Indigenous", "Apparel"] and is_premium:
                    current_weights[i] *= 4
                elif age_grp in ["40s", "50s", "60+"] and is_premium:
                    current_weights[i] *= 3

            sku, u_p, qty, orig_amt, final_amt, p_id = pick_sku_and_amounts(nat, category, sku_pool, current_weights)

            all_transactions.append({
                "tx_id": tx_id,
                "line_no": line_no,
                "tx_time": trans_dt.strftime("%Y-%m-%d %H:%M:%S"),
                "customer_id": cust_id,
                "flight_no": flight["flight_no"],
                "event_id": active_holiday["event_id"] if active_holiday else None,
                "product_sku": sku,
                "qty": qty,
                "unit_price": u_p,
                "net_amount": final_amt,    
                "disc_amount": round(orig_amt - final_amt, 2),
                "promo_id": p_id,
                "payment_method": payment
            })
        tx_count += 1

    return pd.DataFrame(all_transactions)

## 10. SAVE TO CSV + LOAD INTO SQLITE


In [22]:
def save_to_csv(datasets: dict) -> None:
    for name, df in datasets.items():
        path = os.path.join(OUTPUT_DIR, f"{name}.csv")
        df.to_csv(path, index=False, encoding="utf-8-sig")
        print(f"  ✅ {name}.csv  →  {len(df):>5,} rows × {len(df.columns):>2} columns")

In [23]:
def save_to_sqlite(datasets: dict, db_path: str = "duty_free.db") -> None:
    conn = sqlite3.connect(db_path)
    for name, df in datasets.items():
        # drop internal helper columns before saving
        save_df = df.drop(columns=[c for c in ["_date","_dep_dt"] if c in df.columns])
        save_df.to_sql(name, conn, if_exists="replace", index=False)
    conn.close()
    print(f"\n  🗄️  SQLite DB created  →  {db_path}")

## 11. VALIDATION SUMMARY


In [24]:
def print_summary(data_dict) -> None:
    
    if "transactions" not in data_dict:
        print("❌ Error: 'transactions' not found.")
        return
        
    tx_df = data_dict["transactions"].copy()
    cust_df = data_dict["customers"]
    prod_df = data_dict["products"]

    # ── Merge/Join ───────────────────
    df = tx_df.merge(cust_df[['customer_id', 'nationality']], on='customer_id', how='left')
    df = df.merge(prod_df[['product_sku', 'category', 'cost_price']], on='product_sku', how='left')
    
    # ── Pre-processing ────────────────────────────────────────
    id_col   = "tx_id"
    time_col = "tx_time"     
    net_col   = "net_amount"
    disc_col  = "disc_amount"
    
    df[time_col] = pd.to_datetime(df[time_col])
    df['total_cost'] = df['cost_price'] * df['qty']
    df['gross_amount'] = df[net_col] + df[disc_col]

    unique_tx = df.drop_duplicates(id_col)

    print("\n" + "="*65)
    print("  📊 Data Validation Summary (Unified Table)")
    print("="*65)

    # 1. Transaction share by nationality
    print("\n  [Transaction share by nationality]")
    nat_pct = unique_tx["nationality"].value_counts(normalize=True).mul(100).round(1)
    for k, v in nat_pct.head(10).items():
        print(f"    {k:3s} : {v:>5}%")

    # 2. Avg items per basket (Basket Size)
    print("\n  [Avg items per basket by nationality]")
    # Max line_no per transaction represents the basket size
    basket_size = df.groupby(["nationality", "tx_id"])["line_no"].max() \
                    .groupby("nationality").mean().round(2).sort_values(ascending=False)
    for k, v in basket_size.items():
        print(f"    {k:3s} : {v:>4} items")

    # 3. Revenue, Margin, and Markdown by Category
    print("\n  [Revenue, Margin & Markdown by Category]")
    cat_grp = df.groupby("category").agg(
        g_rev = ("gross_amount", "sum"),
        n_rev = (net_col, "sum"),
        cost  = ("total_cost", "sum")  
    )
    cat_grp["rev_share"] = (cat_grp["n_rev"] / cat_grp["n_rev"].sum() * 100).round(1)
    cat_grp["margin_pct"] = ((cat_grp["n_rev"] - cat_grp["cost"]) / cat_grp["n_rev"] * 100).round(1)
    cat_grp["disc_pct"] = ((cat_grp["g_rev"] - cat_grp["n_rev"]) / cat_grp["g_rev"] * 100).round(1)

    for cat, row in cat_grp.sort_values("rev_share", ascending=False).iterrows():
        print(f"    {cat:15s}: Rev {row['rev_share']:>5}% | Margin {row['margin_pct']:>5}% | Disc {row['disc_pct']:>4}%")

    # 4. Average Transaction Value (ATV) 
    print("\n  [Average Transaction Value (ATV)]")
    tx_totals = df.groupby(id_col).agg({net_col: "sum", "nationality": "first"})
    atv_nat = tx_totals.groupby("nationality")[net_col].mean().sort_values(ascending=False).head(3)
    print("    By Nationality (Top 3):")
    for k, v in atv_nat.items():
        print(f"      {k:3s}: AUD {v:>7,.2f}")

    # 5. Holiday event coverage
    print("\n  [Holiday Event Coverage]")
    holiday_txs = df[df["event_id"].notna()]
    h_count = holiday_txs[id_col].nunique()
    total_count = unique_tx[id_col].nunique()
    print(f"    Transactions during holidays: {h_count:,} ({h_count/total_count*100:.1f}%)")
    if h_count > 0:
        event_counts = holiday_txs.drop_duplicates(id_col)["event_id"].value_counts()
        for eid, cnt in event_counts.items():
            print(f"      - {eid:20s}: {cnt:>5,} transactions")

    print("\n" + "="*65)

## 12. MAIN

In [25]:
if __name__ == "__main__":
    print("="*58)
    print("  Sydney T1 Duty-Free — Dataset Generator")
    print("="*58 + "\n")
    print("⏳ Generating data...\n")

    customers                      = generate_customers(8000)
    products                       = generate_products()
    flights                        = generate_flights(weeks=52)
    holiday_events                 = generate_holiday_events()
    transactions = generate_transactions(
        customers, products, flights, holiday_events, n=10000
    )

    flights_clean = flights.drop(columns=["_date","_dep_dt"], errors="ignore")

    datasets = {
        "customer_details"  : customers,
        "product_master"    : products,
        "flight_schedules"  : flights_clean,
        "holiday_events"    : holiday_events,
        "transactions"      : transactions,
    }

    print("📁 Saving CSVs...\n")
    save_to_csv(datasets)

    save_to_sqlite({**datasets, "flight_schedules": flights}, db_path="duty_free.db")

    print_summary({
        "transactions": transactions,
        "customers" : customers,
        "products": products,
        "flight_schedules"  : flights,
    })

    print(f"\n🎉 Done! Output folder : ./{OUTPUT_DIR}/")
    print("   duty_free.db      : SQLite DB for SQL practice")

  Sydney T1 Duty-Free — Dataset Generator

⏳ Generating data...

📁 Saving CSVs...

  ✅ customer_details.csv  →  8,000 rows ×  3 columns
  ✅ product_master.csv  →    108 rows ×  6 columns
  ✅ flight_schedules.csv  →  5,720 rows ×  5 columns
  ✅ holiday_events.csv  →      6 rows ×  4 columns
  ✅ transactions.csv  →  20,219 rows × 13 columns

  🗄️  SQLite DB created  →  duty_free.db

  📊 Data Validation Summary (Unified Table)

  [Transaction share by nationality]
    CN  :  19.9%
    US  :  11.4%
    NZ  :  10.5%
    IN  :   9.9%
    KR  :   8.7%
    JP  :   8.2%
    GB  :   7.2%
    PH  :   7.2%
    TH  :   5.4%
    SG  :   5.1%

  [Avg items per basket by nationality]
    CN  : 2.99 items
    KR  : 2.15 items
    IN  : 2.14 items
    TH  : 2.06 items
    PH  : 2.05 items
    NZ  : 2.03 items
    SG  : 1.98 items
    AU  : 1.97 items
    JP  : 1.97 items
    GB  : 1.91 items
    US  : 1.91 items
    CA  :  1.9 items

  [Revenue, Margin & Markdown by Category]
    Jewellery      : Rev  2

In [26]:
# ── Category Volume vs Revenue ──────────────────────────

validation_df = transactions.merge(products[['product_sku', 'category']], on='product_sku')

category_analysis = validation_df.groupby("category").agg(
    order_count = ("tx_id", "nunique"),    
    total_qty   = ("qty", "sum"),          
    total_rev   = ("net_amount", "sum")    
).sort_values("total_rev", ascending=False)

category_analysis["rev_share"]   = (category_analysis["total_rev"] / category_analysis["total_rev"].sum() * 100).round(1)
category_analysis["volume_share"] = (category_analysis["total_qty"] / category_analysis["total_qty"].sum() * 100).round(1)

print("\n" + "="*75)
print(f"{'Category':<15} | {'Orders':<8} | {'Qty (Vol)':<10} | {'Revenue':<12} | {'Rev %':<7} | {'Vol %':<7}")
print("-"*75)

for cat, row in category_analysis.iterrows():
    print(f"{cat:<15} | {int(row['order_count']):>8,} | {int(row['total_qty']):>10,} | "
          f"AUD {row['total_rev']:>9,.0f} | {row['rev_share']:>5}% | {row['volume_share']:>5}%")
print("="*75)


Category        | Orders   | Qty (Vol)  | Revenue      | Rev %   | Vol %  
---------------------------------------------------------------------------
Jewellery       |    1,643 |      5,672 | AUD   519,671 |  20.7% |   8.4%
Honey           |    1,968 |      7,774 | AUD   428,820 |  17.1% |  11.5%
Liquor          |    1,724 |      6,352 | AUD   417,430 |  16.6% |   9.4%
Tea             |    2,459 |      9,363 | AUD   342,226 |  13.6% |  13.9%
Apparel         |    1,698 |      6,005 | AUD   297,948 |  11.8% |   8.9%
Indigenous      |    1,688 |      5,926 | AUD   146,548 |   5.8% |   8.8%
Souvenir        |    2,079 |      7,647 | AUD   133,572 |   5.3% |  11.3%
Cosmetics       |    2,567 |      8,326 | AUD   129,134 |   5.1% |  12.3%
Confectionery   |    2,614 |     10,527 | AUD    99,551 |   4.0% |  15.6%
